# PropWar Phase A — Targeted Correctness Audit

This executed notebook reads the frozen audit artifacts, rechecks count-based formulas, and reports the acceptance gate. It does not alter public application code or production.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
OUT = ROOT / 'outputs' / 'propwar_correctness_audit'
final = json.loads((OUT / 'final_validation.json').read_text(encoding='utf-8'))
calculations = pd.read_csv(OUT / 'calculation_discrepancies.csv')
home = pd.read_csv(OUT / 'home_validation.csv')
explorer = pd.read_csv(OUT / 'explorer_validation.csv')
findings = pd.read_csv(OUT / 'findings.csv')
final['baseline_commit'], final['source_hashes']

('8b759f18c34708300acf5e3ef84d0e4cbbbde597',
 {'opportunity_events': '9c67605493993ae74191f5ddd865a519afb431caac3611b92e083c90bbbf42d5',
  'canonical_2025': '3c02f26718288adf36b9f7d0759c13722520314998c0d745f1ac76c96535cdfe',
  'situational': 'aec6cd6a11ef36b35bc18ab3468ff6b561951164e75b13b845d5b6d220c88a5c',
  'production': '45f2a601ebe02118c12f93e16dc312a06dacfc8ab8d7f59fec5f64aeb8f06ab2',
  'raw_pbp': '3b7dfe911b842c990f5191f4a911aecac83fcac568eef0df33d720528f0ce32a',
  'raw_schedules': '30711db8aa6e2bbc8e7163a50e8b4eec66cf1af5e38379b85b213cf27862779a',
  'raw_weekly': '023c0f9d4a61c6891a8efd8f8203f4ea494aef29b47343ad493f6c0c1b571403'})

## Coverage and count-based formula verification

In [2]:
displayed = calculations[calculations.displayed_percentage.notna() & calculations.denominator.gt(0)].copy()
displayed['formula_share'] = displayed.numerator / displayed.denominator
formula_max_error = (displayed.formula_share - displayed.expected_percentage).abs().max()
coverage = pd.Series(final['sample_coverage'], name='sample_count').to_frame()
assert formula_max_error < 1e-12
coverage

,sample_count
rb_players,10
wr_players,10
te_players,10
teams,10
games,10
home_rows,25
reports,7
explorer_cases,18


## Window and share results

In [3]:
window_summary = calculations.groupby(['audit_area', 'sample_type', 'status']).size().rename('rows').reset_index()
assert not calculations.loc[calculations.audit_area.eq('Player'), 'status'].eq('FAIL').any()
window_summary

,audit_area,sample_type,status,rows
0,Player,window,PASS,392
1,Teams,role_ownership,PASS,240
2,Teams,situational,FAIL,71
3,Teams,situational,PASS,311


## Home, Explorer, links, and cross-page checks

In [4]:
summary = pd.DataFrame({
    'check': ['Home selected-week rows', 'Explorer zero-inclusive rows', 'Cross-page identical filters', 'Link/state'],
    'failures': [home.status.eq('FAIL').sum(), explorer.status.eq('FAIL').sum(), final['results']['cross_page_failures'], final['results']['link_state_failures']],
})
summary

,check,failures
0,Home selected-week rows,6
1,Explorer zero-inclusive rows,791
2,Cross-page identical filters,0
3,Link/state,7


## Severity and Phase A gate

In [5]:
severity = findings.groupby('severity').size().rename('findings').to_frame()
assert final['phase_status'] == 'FAILED'
assert final['production_status'] == 'UNCHANGED'
assert final['results']['critical_findings'] == 0
severity

,findings
severity,
High,5
Low,1
Medium,3


## Conclusion

Phase A fails because unresolved High correctness issues remain. Production is unchanged. The next authorized action is to wait for the user's screen-recording review.